In [11]:
"""
基于局部视角视频扫描的全局地图绘制
输入: 蛇形扫描的局部视角视频
输出: 全局地图图片

优化说明（v11）：
1. [核心修复] 解决两行长度不一致问题：
   - 新增 align_rows_x_span：在对齐 x_min 之后，还对齐 x_max
   - 对第二行做线性拉伸/压缩，使两行的 X 跨度（x_max - x_min）相等
   - 拉伸方式：以行的 x_min 为锚点，按比例缩放每帧的 tx
2. [保留 v10] 基于全行 x_min 的 X 对齐
3. [保留] 备用位姿估算只修正 Y，保留 tx
4. [保留] Translation_Only=True、亮度归一化、加权融合
"""

import cv2
import numpy as np
import os


# ─────────────────────────────────────────────
# 1. 帧提取
# ─────────────────────────────────────────────
def extract_frames(video_path: str, sample_interval: int = 10,
                   max_frames: int = 2000) -> list:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"无法打开视频: {video_path}")

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    print(f"[视频信息] 总帧数={total}, FPS={fps:.1f}, 时长={total/fps:.1f}s")

    frames, idx = [], 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % sample_interval == 0:
            frames.append(frame)
            if len(frames) >= max_frames:
                break
        idx += 1
    cap.release()
    print(f"[帧提取] 采样帧数: {len(frames)}")
    return frames


# ─────────────────────────────────────────────
# 2. 关键帧筛选
# ─────────────────────────────────────────────
def filter_keyframes(frames: list, diff_threshold: float = 15.0) -> list:
    if not frames:
        return []
    keyframes = [frames[0]]
    for f in frames[1:]:
        prev_gray = cv2.cvtColor(keyframes[-1], cv2.COLOR_BGR2GRAY)
        curr_gray = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
        diff = cv2.absdiff(prev_gray, curr_gray)
        if diff.mean() > diff_threshold:
            keyframes.append(f)
    print(f"[关键帧] 筛选后关键帧数: {len(keyframes)}")
    return keyframes


# ─────────────────────────────────────────────
# 3. 特征检测与匹配
# ─────────────────────────────────────────────
def create_detector(nfeatures: int = 3000):
    try:
        det = cv2.SIFT_create(nfeatures=nfeatures)
        matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
        print("[特征] 使用 SIFT 特征检测器")
        return det, matcher, "SIFT"
    except:
        det = cv2.ORB_create(nfeatures=nfeatures)
        matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
        print("[特征] 使用 ORB 特征检测器")
        return det, matcher, "ORB"


_clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

def enhance_gray(gray: np.ndarray) -> np.ndarray:
    return _clahe.apply(gray)


def match_features(desc1, desc2, matcher, method: str, ratio: float = 0.75) -> list:
    if desc1 is None or desc2 is None:
        return []
    matches = matcher.knnMatch(desc1, desc2, k=2)
    good = []
    for m_pair in matches:
        if len(m_pair) == 2:
            m, n = m_pair
            if m.distance < ratio * n.distance:
                good.append(m)
    return good


# ─────────────────────────────────────────────
# 4. 仿射变换估计
# ─────────────────────────────────────────────
def estimate_homography(kp1, kp2, matches,
                        min_match: int = 10,
                        ransac_thresh: float = 3.0,
                        max_iters: int = 2000,
                        confidence: float = 0.995,
                        max_rotation_deg: float = 1.0,
                        translation_only: bool = True):
    if len(matches) < min_match:
        return None, None
    pts1 = np.float32([kp1[m.queryIdx].pt for m in matches])
    pts2 = np.float32([kp2[m.trainIdx].pt for m in matches])

    M, mask = cv2.estimateAffinePartial2D(
        pts2, pts1,
        method=cv2.RANSAC,
        ransacReprojThreshold=ransac_thresh,
        maxIters=max_iters,
        confidence=confidence
    )
    if M is None or mask.ravel().sum() < min_match:
        return None, None

    inliers = mask.ravel().sum()
    print(f"    匹配点={len(matches)}, 内点={inliers}")

    if translation_only:
        tx, ty = M[0, 2], M[1, 2]
        M = np.array([[1.0, 0.0, tx],
                      [0.0, 1.0, ty]], dtype=np.float64)
    else:
        angle_rad = np.arctan2(M[1, 0], M[0, 0])
        angle_deg = np.degrees(angle_rad)
        if abs(angle_deg) > max_rotation_deg:
            print(f"    [旋转限制] {angle_deg:.2f}° 超阈值，置零")
            scale = np.clip(np.sqrt(M[0, 0]**2 + M[1, 0]**2), 0.95, 1.05)
            M[0, 0] = scale; M[1, 1] = scale
            M[0, 1] = 0.0;   M[1, 0] = 0.0

    H = np.eye(3, dtype=np.float64)
    H[:2, :] = M
    return H, mask


# ─────────────────────────────────────────────
# 5. 换行点检测
# ─────────────────────────────────────────────
def detect_row_breaks(H_global: list,
                      frame_w: int,
                      frame_h: int,
                      x_move_thresh: float = 0.3,
                      y_jump_factor: float = 0.5) -> list:
    tx_list = [H[0, 2] for H in H_global]
    ty_list = [H[1, 2] for H in H_global]

    breaks = []
    window = 3
    for i in range(window, len(H_global) - 1):
        dx_prev = tx_list[i] - tx_list[max(0, i - window)]
        dx_next = tx_list[min(len(tx_list)-1, i + window)] - tx_list[i]
        dy_step = abs(ty_list[i + 1] - ty_list[i])

        x_reversed = (dx_prev * dx_next < 0) and \
                     (abs(dx_prev) > frame_w * x_move_thresh)
        y_jumped = dy_step > frame_h * y_jump_factor

        if (x_reversed or y_jumped) and i not in breaks:
            breaks.append(i)

    breaks = sorted(set(breaks))
    merged = []
    for b in breaks:
        if merged and b - merged[-1] < 5:
            continue
        merged.append(b)

    print(f"[行检测] 检测到 {len(merged)} 个换行点，索引={merged}")
    return merged


# ─────────────────────────────────────────────
# 6. 亮度一致性补偿
# ─────────────────────────────────────────────
def normalize_brightness(keyframes: list, ref_percentile: float = 50.0) -> list:
    global_vals = []
    for f in keyframes[::max(1, len(keyframes)//20)]:
        gray = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
        global_vals.append(np.percentile(gray, ref_percentile))
    global_ref = np.mean(global_vals)

    normalized = []
    for f in keyframes:
        gray = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
        local_val = np.percentile(gray, ref_percentile)
        if local_val > 5:
            gain = np.clip(global_ref / local_val, 0.7, 1.4)
            f_norm = np.clip(f.astype(np.float32) * gain, 0, 255).astype(np.uint8)
        else:
            f_norm = f
        normalized.append(f_norm)

    print(f"[亮度归一化] 全局基准亮度={global_ref:.1f}")
    return normalized


# ─────────────────────────────────────────────
# 7. 位姿平滑
# ─────────────────────────────────────────────
def smooth_poses(H_global: list, row_breaks: list, window: int = 3) -> list:
    segments_idx = []
    prev = 0
    for brk in row_breaks:
        segments_idx.append(list(range(prev, brk + 1)))
        prev = brk + 1
    if prev < len(H_global):
        segments_idx.append(list(range(prev, len(H_global))))

    result = [H.copy() for H in H_global]

    for seg in segments_idx:
        if len(seg) < window * 2 + 1:
            continue
        tx_arr = np.array([H_global[i][0, 2] for i in seg])
        ty_arr = np.array([H_global[i][1, 2] for i in seg])

        kernel = np.exp(-0.5 * np.arange(-window, window+1)**2 / (window/2)**2)
        kernel /= kernel.sum()

        tx_sm = np.convolve(tx_arr, kernel, mode='same')
        ty_sm = np.convolve(ty_arr, kernel, mode='same')
        tx_sm[:window] = tx_arr[:window];   tx_sm[-window:] = tx_arr[-window:]
        ty_sm[:window] = ty_arr[:window];   ty_sm[-window:] = ty_arr[-window:]

        for k, i in enumerate(seg):
            result[i] = H_global[i].copy()
            result[i][0, 2] = tx_sm[k]
            result[i][1, 2] = ty_sm[k]

    print(f"[位姿平滑] 已对 {len(segments_idx)} 段做滑动平均 (window={window})")
    return result


# ─────────────────────────────────────────────
# 8. 行 X 范围计算工具
# ─────────────────────────────────────────────
def get_row_x_bounds(H_global: list, row_seg: list,
                     frame_w: int, frame_h: int) -> tuple:
    """计算一行所有帧投影后的全局 x_min 和 x_max。"""
    corners_base = np.float32([[0, 0], [frame_w, 0],
                                [frame_w, frame_h], [0, frame_h]])
    x_mins, x_maxs = [], []
    for i in row_seg:
        c = cv2.perspectiveTransform(
            corners_base.reshape(-1, 1, 2), H_global[i]
        ).reshape(-1, 2)
        x_mins.append(c[:, 0].min())
        x_maxs.append(c[:, 0].max())
    return float(np.min(x_mins)), float(np.max(x_maxs))


# ─────────────────────────────────────────────
# 9. ✅ v11 核心：行间 X 对齐（同时对齐 x_min 和 x_max）
# ─────────────────────────────────────────────
def align_rows_x_full(H_global: list, row_breaks: list,
                      frame_w: int, frame_h: int,
                      x_align_thresh: float = 0.03) -> list:
    """
    ✅ v11 X 对齐策略：同时对齐两行的 x_min 和 x_max。

    步骤：
    1. 计算参考行（第0行）的 x_min_ref 和 x_max_ref（行宽 span_ref）
    2. 对每个后续行：
       a. 计算其当前 x_min_curr 和 x_max_curr（行宽 span_curr）
       b. 平移修正：tx_offset = x_min_ref - x_min_curr（对齐左端）
       c. 缩放修正：scale = span_ref / span_curr（对齐行宽）
          每帧的 tx 相对行起始点 x_min_curr 做缩放：
          tx_new = x_min_ref + (tx_old - x_min_curr) * scale
       d. 容差控制：仅在偏差超过阈值时执行，缩放比例限制在 [0.85, 1.15]

    注意：Y 方向不修正。
    """
    row_segments = []
    prev = 0
    for brk in row_breaks:
        row_segments.append(list(range(prev, brk + 1)))
        prev = brk + 1
    if prev < len(H_global):
        row_segments.append(list(range(prev, len(H_global))))

    if len(row_segments) < 2:
        return H_global

    result = [H.copy() for H in H_global]

    # 参考行（第0行）的 X 范围
    x_min_ref, x_max_ref = get_row_x_bounds(result, row_segments[0], frame_w, frame_h)
    span_ref = x_max_ref - x_min_ref
    print(f"[X对齐] 参考行(第0行) x_min={x_min_ref:.1f}, x_max={x_max_ref:.1f}, 跨度={span_ref:.1f}px")

    for row_idx in range(1, len(row_segments)):
        curr_seg = row_segments[row_idx]

        x_min_curr, x_max_curr = get_row_x_bounds(result, curr_seg, frame_w, frame_h)
        span_curr = x_max_curr - x_min_curr

        x_diff = x_min_ref - x_min_curr          # 左端偏差
        span_diff = abs(span_ref - span_curr)     # 行宽偏差

        print(f"[X对齐] 第{row_idx}行 x_min={x_min_curr:.1f}, x_max={x_max_curr:.1f}, "
              f"跨度={span_curr:.1f}px | 左偏差={x_diff:.1f}, 跨度差={span_diff:.1f}")

        # ── 步骤1：平移对齐左端 ──
        if abs(x_diff) > frame_w * x_align_thresh:
            print(f"    → 执行左端平移修正 {x_diff:.1f}px")
            for i in curr_seg:
                result[i] = result[i].copy()
                result[i][0, 2] += x_diff
            # 更新 x_min_curr
            x_min_curr += x_diff
            x_max_curr += x_diff

        # ── 步骤2：缩放对齐行宽 ──
        if span_curr > 1.0 and span_diff > frame_w * x_align_thresh:
            scale = span_ref / span_curr
            scale = np.clip(scale, 0.85, 1.15)  # 限制缩放幅度
            print(f"    → 执行行宽缩放修正 scale={scale:.4f}")
            for i in curr_seg:
                old_tx = result[i][0, 2]
                # 以 x_min_curr 为锚点做缩放
                result[i][0, 2] = x_min_curr + (old_tx - x_min_curr) * scale
        else:
            print(f"    → 行宽偏差={span_diff:.1f}px，在阈值内，无需缩放")

    return result


# ─────────────────────────────────────────────
# 10. 备用位姿估算（只修正 Y，保留 tx）
# ─────────────────────────────────────────────
def estimate_row_start_pose_fallback(H_global: list, brk_idx: int,
                                      next_frame_i: int,
                                      frame_w: int, frame_h: int) -> np.ndarray:
    """
    跨行特征匹配失败时的备用方案：
    只把新行第一帧的 ty 修正为换行点帧底部边界，tx 保持不变。
    后续 align_rows_x_full 会再做 X 精修。
    """
    H_brk = H_global[brk_idx]
    H_next = H_global[next_frame_i]

    corners_base = np.float32([[0, 0], [frame_w, 0],
                                [frame_w, frame_h], [0, frame_h]])
    c_brk = cv2.perspectiveTransform(
        corners_base.reshape(-1, 1, 2), H_brk
    ).reshape(-1, 2)
    brk_y_max = c_brk[:, 1].max()

    H_estimated = H_next.copy()
    H_estimated[1, 2] = brk_y_max
    print(f"    [备用估算] 保留 tx={H_next[0,2]:.1f}，修正 ty={brk_y_max:.1f}")
    return H_estimated


# ─────────────────────────────────────────────
# 11. 渐进式全局地图拼接（v11 核心）
# ─────────────────────────────────────────────
def stitch_frames(keyframes: list,
                  output_size_limit: int = 8000,
                  look_back: int = 8,
                  nfeatures: int = 3000,
                  ratio: float = 0.75,
                  min_match: int = 10,
                  ransac_thresh: float = 3.0,
                  max_iters: int = 2000,
                  confidence: float = 0.995,
                  edge_blend_width: float = 40.0,
                  smooth_window: int = 3,
                  max_rotation_deg: float = 1.0,
                  translation_only: bool = True,
                  cross_anchor_range: int = 15,
                  x_align_thresh: float = 0.03) -> np.ndarray:

    keyframes = normalize_brightness(keyframes)
    det, matcher, method = create_detector(nfeatures)

    print("[特征提取] 计算所有关键帧特征点（CLAHE 增强）...")
    kps, descs = [], []
    for i, f in enumerate(keyframes):
        gray = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
        gray = enhance_gray(gray)
        kp, desc = det.detectAndCompute(gray, None)
        kps.append(kp)
        descs.append(desc)
        if (i + 1) % 10 == 0:
            print(f"  已处理 {i+1}/{len(keyframes)} 帧")

    h_frame, w_frame = keyframes[0].shape[:2]

    # ── 第一遍：标准多锚帧匹配 ──
    H_global = [np.eye(3, dtype=np.float64)]
    print("[单应矩阵] 第一遍：标准多锚帧匹配...")
    for i in range(1, len(keyframes)):
        best_H, best_inliers = None, 0
        for j in range(max(0, i - look_back), i):
            good = match_features(descs[j], descs[i], matcher, method, ratio)
            H_local, mask = estimate_homography(
                kps[j], kps[i], good,
                min_match, ransac_thresh, max_iters, confidence,
                max_rotation_deg=max_rotation_deg,
                translation_only=translation_only
            )
            if H_local is None:
                continue
            inliers = mask.ravel().sum()
            if inliers > best_inliers:
                best_inliers = inliers
                best_H = H_global[j] @ H_local
        H_global.append(best_H if best_H is not None else H_global[-1].copy())

    # ── 检测换行点 ──
    row_breaks = detect_row_breaks(H_global, frame_w=w_frame, frame_h=h_frame)
    print(f"[行检测] row_breaks = {row_breaks}")

    # ── 第二遍：跨行强制匹配，并传播修正 ──
    print("[单应矩阵] 第二遍：跨行强制匹配（含行内传播修正）...")
    for brk_idx_pos, brk_idx in enumerate(row_breaks):
        next_frame_i = brk_idx + 1
        if next_frame_i >= len(keyframes):
            continue

        print(f"  处理换行点 brk={brk_idx}, 下一帧={next_frame_i}")
        best_H, best_inliers = None, 0

        actual_cross_range = min(cross_anchor_range, brk_idx + 1)
        anchor_range = range(max(0, brk_idx - actual_cross_range + 1), brk_idx + 1)

        target_frames = [next_frame_i]
        for extra in [1, 2, 3]:
            if next_frame_i + extra < len(keyframes):
                target_frames.append(next_frame_i + extra)

        for target_i in target_frames:
            for j in anchor_range:
                good = match_features(descs[j], descs[target_i], matcher, method, ratio)
                H_local, mask = estimate_homography(
                    kps[j], kps[target_i], good,
                    min_match, ransac_thresh, max_iters, confidence,
                    max_rotation_deg=max_rotation_deg,
                    translation_only=translation_only
                )
                if H_local is None:
                    continue
                inliers = mask.ravel().sum()
                if inliers > best_inliers:
                    best_inliers = inliers
                    H_target = H_global[j] @ H_local
                    if target_i == next_frame_i:
                        best_H = H_target
                    else:
                        H_rel_old = np.linalg.inv(H_global[next_frame_i]) @ H_global[target_i]
                        best_H = H_target @ np.linalg.inv(H_rel_old)

        row_end = row_breaks[brk_idx_pos + 1] if brk_idx_pos + 1 < len(row_breaks) else len(keyframes)

        if best_H is not None and best_inliers >= min_match:
            print(f"    跨行匹配成功，内点={best_inliers}，更新 H_global[{next_frame_i}]")
            H_correction = best_H @ np.linalg.inv(H_global[next_frame_i])
            for k in range(next_frame_i, row_end):
                H_global[k] = H_correction @ H_global[k]
            print(f"    已将修正传播到帧 {next_frame_i}~{row_end-1}")
        else:
            print(f"    跨行匹配失败（内点={best_inliers}），使用备用估算...")
            H_estimated = estimate_row_start_pose_fallback(
                H_global, brk_idx, next_frame_i, w_frame, h_frame
            )
            H_correction = H_estimated @ np.linalg.inv(H_global[next_frame_i])
            for k in range(next_frame_i, row_end):
                H_global[k] = H_correction @ H_global[k]
            print(f"    备用估算已传播到帧 {next_frame_i}~{row_end-1}")

    # ── ✅ v11: 同时对齐 x_min 和 x_max（解决行长不一致问题）──
    H_global = align_rows_x_full(
        H_global, row_breaks, w_frame, h_frame,
        x_align_thresh=x_align_thresh
    )

    # ── 位姿平滑 ──
    H_global = smooth_poses(H_global, row_breaks, window=smooth_window)

    # ── 计算全局画布尺寸 ──
    h, w = keyframes[0].shape[:2]
    corners_base = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    all_corners = []
    for H in H_global:
        c = cv2.perspectiveTransform(corners_base.reshape(-1, 1, 2), H).reshape(-1, 2)
        all_corners.append(c)
    all_corners = np.vstack(all_corners)

    x_min, y_min = all_corners.min(axis=0)
    x_max, y_max = all_corners.max(axis=0)
    canvas_w = int(np.ceil(x_max - x_min))
    canvas_h = int(np.ceil(y_max - y_min))

    scale = 1.0
    if canvas_w > output_size_limit or canvas_h > output_size_limit:
        scale = output_size_limit / max(canvas_w, canvas_h)
        canvas_w = int(canvas_w * scale)
        canvas_h = int(canvas_h * scale)
        print(f"[画布缩放] 缩放比例={scale:.3f}")

    print(f"[画布] 尺寸: {canvas_w} x {canvas_h} px")

    T = np.array([[scale, 0, -x_min * scale],
                  [0, scale, -y_min * scale],
                  [0, 0, 1]], dtype=np.float64)

    # ── 构建每帧所属行号 ──
    frame_row = []
    current_row = 0
    break_set = set(row_breaks)
    for i in range(len(keyframes)):
        frame_row.append(current_row)
        if i in break_set:
            current_row += 1

    # ── 距离变换加权融合 ──
    canvas_sum    = np.zeros((canvas_h, canvas_w, 3), dtype=np.float64)
    canvas_weight = np.zeros((canvas_h, canvas_w),    dtype=np.float64)

    print("[拼接] 加权叠加融合...")
    for i, (frame, H) in enumerate(zip(keyframes, H_global)):
        H_final = T @ H

        warped = cv2.warpPerspective(
            frame.astype(np.float32), H_final,
            (canvas_w, canvas_h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT, borderValue=0
        )
        raw_mask = cv2.warpPerspective(
            np.ones((h, w), dtype=np.float32), H_final,
            (canvas_w, canvas_h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT, borderValue=0
        )
        frame_valid = (raw_mask > 0.5)

        dist = cv2.distanceTransform(
            frame_valid.astype(np.uint8), cv2.DIST_L2, 5
        ).astype(np.float64)

        is_row_start = (i > 0 and frame_row[i] != frame_row[i - 1])
        blend_w = edge_blend_width * 2.5 if is_row_start else edge_blend_width
        weight = np.clip(dist / blend_w, 0.0, 1.0)

        canvas_sum   += warped.astype(np.float64) * weight[:, :, np.newaxis]
        canvas_weight += weight

        if (i + 1) % 10 == 0:
            print(f"  已融合 {i+1}/{len(keyframes)} 帧")

    # ── 归一化 ──
    mask_nonzero = canvas_weight > 0
    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.float32)
    canvas[mask_nonzero] = (
        canvas_sum[mask_nonzero] / canvas_weight[mask_nonzero, np.newaxis]
    ).astype(np.float32)

    # ── 接缝区域模糊平滑 ──
    result_uint8 = np.clip(canvas, 0, 255).astype(np.uint8)
    seam_region = (canvas_weight > 0) & (canvas_weight < edge_blend_width * 0.5)
    if seam_region.sum() > 100:
        blurred = cv2.GaussianBlur(result_uint8, (5, 5), 1.5)
        seam_mask = seam_region.astype(np.float32)[:, :, np.newaxis]
        result_uint8 = np.clip(
            result_uint8.astype(np.float32) * (1 - seam_mask) +
            blurred.astype(np.float32) * seam_mask,
            0, 255
        ).astype(np.uint8)

    return result_uint8


# ─────────────────────────────────────────────
# 12. 后处理
# ─────────────────────────────────────────────
def post_process(img: np.ndarray,
                 sharpen_center: float = 2.2,
                 sharpen_edge: float = 0.3) -> np.ndarray:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 5, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(thresh)
    if coords is None:
        return img
    x, y, w, h = cv2.boundingRect(coords)
    cropped = img[y:y+h, x:x+w]
    kernel = np.array([[0, -sharpen_edge, 0],
                       [-sharpen_edge, sharpen_center, -sharpen_edge],
                       [0, -sharpen_edge, 0]])
    return cv2.filter2D(cropped, -1, kernel)


# ─────────────────────────────────────────────
# 13. 备用方案
# ─────────────────────────────────────────────
def try_opencv_stitcher(frames: list):
    print("[备用] 尝试 OpenCV 内置 Stitcher...")
    step = max(1, len(frames) // 50)
    subset = frames[::step][:50]
    stitcher = cv2.Stitcher_create(cv2.Stitcher_SCANS)
    status, pano = stitcher.stitch(subset)
    if status == cv2.Stitcher_OK:
        print("[备用] Stitcher 成功！")
        return pano
    codes = {1: "ERR_NEED_MORE_IMGS", 2: "ERR_HOMOGRAPHY_EST_FAIL",
             3: "ERR_CAMERA_PARAMS_ADJUST_FAIL"}
    print(f"[备用] Stitcher 失败: {codes.get(status, status)}")
    return None


# ─────────────────────────────────────────────
# 主流程
# ─────────────────────────────────────────────
def build_map(video_path: str,
              output_path: str = "global_map.png",
              sample_interval: int = 3,
              max_frames: int = 2000,
              diff_threshold: float = 10.0,
              use_builtin_stitcher: bool = False,
              nfeatures: int = 3000,
              ratio: float = 0.75,
              look_back: int = 8,
              min_match: int = 10,
              ransac_thresh: float = 3.0,
              max_iters: int = 2000,
              confidence: float = 0.995,
              edge_blend_width: float = 40.0,
              smooth_window: int = 3,
              output_size_limit: int = 8000,
              sharpen_center: float = 2.2,
              sharpen_edge: float = 0.3,
              max_rotation_deg: float = 1.0,
              translation_only: bool = True,
              cross_anchor_range: int = 15,
              x_align_thresh: float = 0.03) -> str:
    print("=" * 55)
    print("  全局地图绘制 — 局部视角视频 → 全景地图  v11")
    print("=" * 55)

    frames = extract_frames(video_path, sample_interval, max_frames)
    if len(frames) < 3:
        raise ValueError("帧数不足，请检查视频文件")

    result = None
    if use_builtin_stitcher:
        result = try_opencv_stitcher(frames)

    if result is None:
        keyframes = filter_keyframes(frames, diff_threshold)
        if len(keyframes) < 3:
            print("[警告] 关键帧过少，降低 diff_threshold 重试")
            keyframes = filter_keyframes(frames, diff_threshold / 2)
        result = stitch_frames(
            keyframes,
            output_size_limit=output_size_limit,
            look_back=look_back,
            nfeatures=nfeatures,
            ratio=ratio,
            min_match=min_match,
            ransac_thresh=ransac_thresh,
            max_iters=max_iters,
            confidence=confidence,
            edge_blend_width=edge_blend_width,
            smooth_window=smooth_window,
            max_rotation_deg=max_rotation_deg,
            translation_only=translation_only,
            cross_anchor_range=cross_anchor_range,
            x_align_thresh=x_align_thresh,
        )

    print("[后处理] 裁剪 & 锐化...")
    result = post_process(result, sharpen_center, sharpen_edge)

    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    cv2.imwrite(output_path, result)
    h, w = result.shape[:2]
    size_kb = os.path.getsize(output_path) // 1024
    print(f"\n✅ 全局地图已保存: {output_path}")
    print(f"   图像尺寸: {w} x {h} px  |  文件大小: {size_kb} KB")
    return output_path


# ── 路径 ──────────────────────────────────────────────
Scan_Video_Path = r'F:\0_My_jupyter\0_Optical\scan_output\Target_3_layer.mp4'
Lingo_Map_Path  = r'F:\0_My_jupyter\0_Optical\scan_output\global_map_test_auto_4.png'

# ── 帧提取 ────────────────────────────────────────────
Interval    = 2
Max_Frames  = 2000

# ── 关键帧筛选 ────────────────────────────────────────
Diff        = 10.0

# ── 特征检测 ──────────────────────────────────────────
NFeatures   = 10000

# ── 特征匹配 ──────────────────────────────────────────
Ratio       = 0.75

# ── 多锚帧回溯 ────────────────────────────────────────
Look_Back   = 8

# ── RANSAC 单应矩阵估计 ───────────────────────────────
Min_Match       = 10
Ransac_Thresh   = 4.0
Max_Iters       = 2000
Confidence      = 0.995

# ── v11 关键参数 ──────────────────────────────────────
# 纯平移模式（True = 完全禁止旋转/缩放）
Translation_Only   = True

# 旋转限制（Translation_Only=False 时才生效）
Max_Rotation_Deg   = 1.0

# 跨行匹配回溯帧数
Cross_Anchor_Range = 15

# X 对齐阈值（相对帧宽）：
# v11 同时对齐 x_min 和 x_max，0.03 = 3% 帧宽即触发
X_Align_Thresh     = 0.03

# ── 融合 ──────────────────────────────────────────────
Edge_Blend_Width = 40.0
Smooth_Window   = 3

# ── 画布输出 ──────────────────────────────────────────
Output_Size_Limit = 10000

# ── 后处理锐化 ────────────────────────────────────────
Sharpen_Center  = 2.2
Sharpen_Edge    = 0.35

# ── 备用拼接器 ────────────────────────────────────────
Builtin = False

# ═══════════════════════════════════════════════════════

build_map(
    video_path        = Scan_Video_Path,
    output_path       = Lingo_Map_Path,
    sample_interval   = Interval,
    max_frames        = Max_Frames,
    diff_threshold    = Diff,
    use_builtin_stitcher = Builtin,
    nfeatures         = NFeatures,
    ratio             = Ratio,
    look_back         = Look_Back,
    min_match         = Min_Match,
    ransac_thresh     = Ransac_Thresh,
    max_iters         = Max_Iters,
    confidence        = Confidence,
    edge_blend_width  = Edge_Blend_Width,
    smooth_window     = Smooth_Window,
    output_size_limit = Output_Size_Limit,
    sharpen_center    = Sharpen_Center,
    sharpen_edge      = Sharpen_Edge,
    max_rotation_deg  = Max_Rotation_Deg,
    translation_only  = Translation_Only,
    cross_anchor_range = Cross_Anchor_Range,
    x_align_thresh    = X_Align_Thresh,
)

  全局地图绘制 — 局部视角视频 → 全景地图  v11
[视频信息] 总帧数=207, FPS=24.0, 时长=8.6s
[帧提取] 采样帧数: 104
[关键帧] 筛选后关键帧数: 88
[亮度归一化] 全局基准亮度=172.7
[特征] 使用 SIFT 特征检测器
[特征提取] 计算所有关键帧特征点（CLAHE 增强）...
  已处理 10/88 帧
  已处理 20/88 帧
  已处理 30/88 帧
  已处理 40/88 帧
  已处理 50/88 帧
  已处理 60/88 帧
  已处理 70/88 帧
  已处理 80/88 帧
[单应矩阵] 第一遍：标准多锚帧匹配...
    匹配点=190, 内点=147
    匹配点=150, 内点=79
    匹配点=427, 内点=170
    匹配点=111, 内点=77
    匹配点=320, 内点=231
    匹配点=366, 内点=148
    匹配点=61, 内点=24
    匹配点=192, 内点=55
    匹配点=194, 内点=100
    匹配点=463, 内点=118
    匹配点=106, 内点=71
    匹配点=157, 内点=54
    匹配点=238, 内点=104
    匹配点=299, 内点=56
    匹配点=125, 内点=28
    匹配点=146, 内点=24
    匹配点=232, 内点=35
    匹配点=134, 内点=28
    匹配点=156, 内点=17
    匹配点=263, 内点=77
    匹配点=388, 内点=64
    匹配点=213, 内点=23
    匹配点=360, 内点=57
    匹配点=218, 内点=17
    匹配点=297, 内点=71
    匹配点=1575, 内点=826
    匹配点=304, 内点=30
    匹配点=318, 内点=49
    匹配点=257, 内点=18
    匹配点=270, 内点=29
    匹配点=421, 内点=90
    匹配点=217, 内点=20
    匹配点=264, 内点=43
    匹配点=201, 内点=15
    匹配点=220, 内点=18
    匹配点=355, 内点=85
    匹

'F:\\0_My_jupyter\\0_Optical\\scan_output\\global_map_test_auto_4.png'